In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
import sys
from pathlib import Path

# Update this to the repo location in your Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/mlsd-for-duckitown")
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "PROJECT_ROOT not found. Update the path to your repo in Google Drive."
    )

os.chdir(PROJECT_ROOT)
%pip install -r requirements.txt

FINE_TUNING_DIR = PROJECT_ROOT / "fine-tuning"

# Dataset paths (fixed layout under PROJECT_ROOT)
LABEL_FILE_NAME = "_annotation.wireframe.json"
DATASET_ROOT = PROJECT_ROOT / "dataset"
TRAIN_IMAGE_DIR = DATASET_ROOT / "train"
TRAIN_LABEL_PATH = TRAIN_IMAGE_DIR / LABEL_FILE_NAME
VAL_IMAGE_DIR = DATASET_ROOT / "valid"
VAL_LABEL_PATH = VAL_IMAGE_DIR / LABEL_FILE_NAME
TEST_IMAGE_DIR = DATASET_ROOT / "test"
TEST_LABEL_PATH = TEST_IMAGE_DIR / LABEL_FILE_NAME

In [27]:
# Verify TensorFlow sees a GPU
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())

physical_gpus = tf.config.list_physical_devices("GPU")
print("Physical GPUs:", physical_gpus)

if physical_gpus:
    try:
        for gpu in physical_gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices("GPU")
        print("Logical GPUs:", logical_gpus)
        print("GPU is available and configured.")
    except RuntimeError as exc:
        print("GPU detected but could not set memory growth:", exc)
else:
    print("No GPU detected. Check Colab runtime settings.")


TensorFlow version: 2.20.0
Built with CUDA: True
Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Logical GPUs: [LogicalDevice(name='/device:GPU:0', device_type='GPU')]
GPU is available and configured.


In [29]:
from dataclasses import dataclass

@dataclass
class TrainConfig:
    model_type: str = "M-LSD_512_large"
    ckpt_dir: str = ""
    save_dir: str = ""

    batch_size: int = 16
    epochs: int = 100
    learning_rate: float = 0.001

    warmup_epochs: int = 5
    decay_start_epoch: int = 50
    eval_every: int = 1

    loss_weight_cls: float = 1.0
    loss_weight_reg: float = 1.0
    augment: bool = True

config = TrainConfig()
config.ckpt_dir=str(PROJECT_ROOT / "ckpt_models" / f"{config.model_type}")
config.save_dir=str(PROJECT_ROOT / "fine-tuned_model" / f"{config.model_type}_ft_{config.epochs}_0001")
os.makedirs(config.save_dir, exist_ok=True)

In [30]:
os.chdir(FINE_TUNING_DIR)
%pwd

'/content/drive/MyDrive/Colab Notebooks/mlsd-for-duckitown/fine-tuning'

In [31]:
import json
import math
from dataclasses import dataclass
from typing import List, Tuple

import tensorflow as tf

import dataloader as dl
import load_model


def setup_gpu() -> None:
    gpus = tf.config.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.keras.mixed_precision.set_global_policy("mixed_float16")


def load_labels(label_path: str, image_dir: str) -> Tuple[List[str], List[List[List[float]]]]:
    if not os.path.isfile(label_path):
        raise FileNotFoundError(f"Label json not found: {label_path}")

    with open(label_path, "r", encoding="utf-8") as f:
        labels = json.load(f)

    image_paths = []
    lines_list = []
    for item in labels:
        filename = item.get("filename")
        lines = item.get("lines", [])
        if not filename:
            continue
        image_paths.append(os.path.join(image_dir, filename))
        lines_list.append(lines)

    if not image_paths:
        raise ValueError("No labels found in label json.")

    return image_paths, lines_list


def get_lr(epoch: int, config: TrainConfig) -> float:
    if epoch < config.warmup_epochs:
        return config.learning_rate * (epoch / max(1, config.warmup_epochs))
    if epoch < config.decay_start_epoch:
        return config.learning_rate

    progress = (epoch - config.decay_start_epoch) / max(1, config.epochs - config.decay_start_epoch)
    return 0.5 * config.learning_rate * (1.0 + math.cos(math.pi * progress))


def _masked_huber(
    y_true: tf.Tensor,
    y_pred: tf.Tensor,
    mask: tf.Tensor,
    huber: tf.keras.losses.Huber,
) -> tf.Tensor:
    true_vals = tf.boolean_mask(y_true, mask)
    pred_vals = tf.boolean_mask(y_pred, mask)
    return tf.cond(
        tf.size(true_vals) > 0,
        lambda: tf.reduce_mean(huber(true_vals, pred_vals)),
        lambda: tf.constant(0.0, dtype=tf.float32),
    )


def compute_loss(y_true: tf.Tensor, y_pred: tf.Tensor, config: TrainConfig) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    cls_idx = [0, 7, 14, 15]
    y_true_cls = tf.gather(y_true, cls_idx, axis=-1)
    y_pred_cls = tf.gather(y_pred, cls_idx, axis=-1)

    bce = tf.keras.losses.BinaryCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
    cls_loss = tf.reduce_mean(bce(y_true_cls, y_pred_cls))

    huber = tf.keras.losses.Huber(reduction=tf.keras.losses.Reduction.NONE)
    mask_a = y_true[..., 0] > 0.0
    mask_b = y_true[..., 7] > 0.0

    reg_loss_a = _masked_huber(y_true[..., 1:7], y_pred[..., 1:7], mask_a, huber)
    reg_loss_b = _masked_huber(y_true[..., 8:14], y_pred[..., 8:14], mask_b, huber)
    reg_loss = reg_loss_a + reg_loss_b

    total_loss = (cls_loss * config.loss_weight_cls) + (reg_loss * config.loss_weight_reg)
    return total_loss, cls_loss, reg_loss


setup_gpu()

model_cfg = load_model.infer_config_from_path(config.ckpt_dir)
model_cfg.batch_size = config.batch_size
model = load_model.load_pretrained_model(model_cfg, config.ckpt_dir, return_train_map=True)


image_paths, lines_list = load_labels(str(TRAIN_LABEL_PATH), str(TRAIN_IMAGE_DIR))
dataset = dl.build_dataloader(
    image_paths=image_paths,
    lines_list=lines_list,
    batch_size=config.batch_size,
    target_size=model_cfg.input_size,
    augment=config.augment,
 )

val_dataset = None
if os.path.isfile(str(VAL_LABEL_PATH)):
    val_image_paths, val_lines_list = load_labels(str(VAL_LABEL_PATH), str(VAL_IMAGE_DIR))
    val_dataset = dl.build_dataloader(
        image_paths=val_image_paths,
        lines_list=val_lines_list,
        batch_size=config.batch_size,
        target_size=model_cfg.input_size,
        augment=False,
    )

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0)
if tf.keras.mixed_precision.global_policy().compute_dtype == "float16":
    optimizer = tf.keras.mixed_precision.LossScaleOptimizer(optimizer)

ckpt = tf.train.Checkpoint(step=tf.Variable(0, name="step"), optimizer=optimizer, model=model)
manager = tf.train.CheckpointManager(ckpt, config.save_dir, max_to_keep=1)


@tf.function
def train_step(images: tf.Tensor, targets: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
    with tf.GradientTape() as tape:
        preds = model(images, training=True)
        total_loss, cls_loss, reg_loss = compute_loss(targets, preds, config)

        if isinstance(optimizer, tf.keras.mixed_precision.LossScaleOptimizer):
            scaled_loss = optimizer.scale_loss(total_loss)
        else:
            scaled_loss = total_loss

    grads = tape.gradient(scaled_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return total_loss, cls_loss, reg_loss


@tf.function
def eval_step(images: tf.Tensor, targets: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
    preds = model(images, training=False)
    total_loss, cls_loss, reg_loss = compute_loss(targets, preds, config)
    return total_loss, cls_loss, reg_loss


Model: "WireFrameModel"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ input_image           │ (None, 512, 512,  │           0 │ -                  │
│ (InputLayer)          │ 3)                │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ true_divide           │ (None, 512, 512,  │           0 │ input_image[0][0]  │
│ (TrueDivide)          │ 3)                │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ subtract (Subtract)   │ (None, 512, 512,  │           0 │ true_divide[0][0]  │
│                       │ 3)                │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ MLSD_large_extractor  │ [(None, 256, 256, │     558,656 │ subtract[0][0]     │
│ (Functional)          │ 16), (None, 128,  │             │                    │
│                       │ 128, 24), (None,  │             │                    │
│                       │ 64, 64, 32),      │             │                    │
│                       │ (None, 32, 32,    │             │                    │
│                       │ 64), (None, 32,   │             │                    │
│                       │ 32, 96)]          │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ decoder_fpn           │ (None, 256, 256,  │     916,992 │ MLSD_large_extrac… │
│ (Decoder_FPN)         │ 64)               │             │ MLSD_large_extrac… │
│                       │                   │             │ MLSD_large_extrac… │
│                       │                   │             │ MLSD_large_extrac… │
│                       │                   │             │ MLSD_large_extrac… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ Decoder (Decoder)     │ [(None, 256, 256, │      75,280 │ decoder_fpn[0][0]  │
│                       │ 1), (None, 256,   │             │                    │
│                       │ 256, 4), (None,   │             │                    │
│                       │ 200, 2), (None,   │             │                    │
│                       │ 200), (None, 256, │             │                    │
│                       │ 256, 4), (None,   │             │                    │
│                       │ 256, 256, 1),     │             │                    │
│                       │ (None, 256, 256,  │             │                    │
│                       │ 1), (None, 200,   │             │                    │
│                       │ 2), (None, 200),  │             │                    │
│                       │ (None, 256, 256,  │             │                    │
│                       │ 1), (None, 256,   │             │                    │
│                       │ 256, 4), (None,   │             │                    │
│                       │ 200, 2), (None,   │             │                    │
│                       │ 200), (None, 256, │             │                    │
│                       │ 256, 1), (None,   │             │                    │
│                       │ 256, 256, 1),     │             │                    │
│                       │ (None, 256, 256,  │             │                    │
│                       │ 1), (None, 256,   │             │                    │
│                       │ 256, 1)]          │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ get_item (GetItem)    │ (None, 256, 256,  │           0 │ Decoder[0][10]     │
│                       │ 2)                │             │                    │
├───────────────────────┼──────

 Total params: 1,550,928 (5.92 MB)

 Trainable params: 1,531,984 (5.84 MB)

 Non-trainable params: 18,944 (74.00 KB)

[*] load ckpt from /content/drive/MyDrive/Colab Notebooks/mlsd-for-duckitown/ckpt_models/M-LSD_512_large/ckpt-151 at step 58500.


In [ ]:
for epoch in range(config.epochs):
    lr = get_lr(epoch, config)
    optimizer.learning_rate.assign(lr)

    total_meter = tf.keras.metrics.Mean()
    cls_meter = tf.keras.metrics.Mean()
    reg_meter = tf.keras.metrics.Mean()

    for images, targets in dataset:
        total_loss, cls_loss, reg_loss = train_step(images, targets)
        total_meter.update_state(total_loss)
        cls_meter.update_state(cls_loss)
        reg_meter.update_state(reg_loss)

    log = (
        f"Epoch {epoch + 1}/{config.epochs} | lr={lr:.6f} | "
        f"loss={total_meter.result():.4f} | "
        f"cls={cls_meter.result():.4f} | reg={reg_meter.result():.4f}"
    )

    if val_dataset is not None and (epoch + 1) % config.eval_every == 0:
        val_total = tf.keras.metrics.Mean()
        val_cls = tf.keras.metrics.Mean()
        val_reg = tf.keras.metrics.Mean()
        for images, targets in val_dataset:
            total_loss, cls_loss, reg_loss = eval_step(images, targets)
            val_total.update_state(total_loss)
            val_cls.update_state(cls_loss)
            val_reg.update_state(reg_loss)
        log += (
            f" | val_loss={val_total.result():.4f}"
            f" | val_cls={val_cls.result():.4f}"
            f" | val_reg={val_reg.result():.4f}"
        )

    print(log)

    ckpt.step.assign_add(1)

final_checkpoint = manager.save()
print("Saved final checkpoint to:", final_checkpoint)

Epoch 1/100 | lr=0.000000 | loss=1.7424 | cls=0.7335 | reg=1.0089 | val_loss=1.0794 | val_cls=0.6563 | val_reg=0.4231
Epoch 2/100 | lr=0.000200 | loss=1.2365 | cls=0.6898 | reg=0.5467 | val_loss=1.9877 | val_cls=1.5627 | val_reg=0.4251
Epoch 3/100 | lr=0.000400 | loss=0.8482 | cls=0.5541 | reg=0.2941 | val_loss=0.8466 | val_cls=0.5163 | val_reg=0.3302
Epoch 4/100 | lr=0.000600 | loss=0.6885 | cls=0.4166 | reg=0.2719 | val_loss=0.7361 | val_cls=0.4169 | val_reg=0.3192
Epoch 5/100 | lr=0.000800 | loss=0.5661 | cls=0.3125 | reg=0.2536 | val_loss=0.5996 | val_cls=0.2659 | val_reg=0.3337
Epoch 6/100 | lr=0.001000 | loss=0.4690 | cls=0.2301 | reg=0.2389 | val_loss=0.5230 | val_cls=0.2252 | val_reg=0.2978
Epoch 7/100 | lr=0.001000 | loss=0.4127 | cls=0.1772 | reg=0.2355 | val_loss=0.5050 | val_cls=0.2122 | val_reg=0.2929
Epoch 8/100 | lr=0.001000 | loss=0.3751 | cls=0.1420 | reg=0.2331 | val_loss=0.4762 | val_cls=0.2119 | val_reg=0.2643
Epoch 9/100 | lr=0.001000 | loss=0.3494 | cls=0.1173 | r

In [ ]:
from dataclasses import dataclass

# --- Edit test config here (dataset paths are fixed in the setup cell) ---


@dataclass
class TestConfig:
    """Checkpoint path and inference settings for evaluation / visualization."""

    model_path: str = str(PROJECT_ROOT / "fine-tuned_model" / "M-LSD_512_large_ft_100_0001")

    input_size: int = 512
    backbone_type: str = "MLSD_large"

    num_samples: int = 5
    score_thr: float = 0.5
    dist_thr: float = 10
    topk: int = 200

    loss_weight_cls: float = 1.0
    loss_weight_reg: float = 1.0


pretrained_test_config = TestConfig(
    model_path=str(PROJECT_ROOT / "ckpt_models" / config.model_type),
)

finetuned_test_config = TestConfig(
    model_path=config.save_dir,
)

test_config = finetuned_test_config

print("Pretrained:", pretrained_test_config)
print("Finetuned:", finetuned_test_config)

Pretrained: TestConfig(model_path='/content/drive/MyDrive/Colab Notebooks/mlsd-for-duckitown/ckpt_models/M-LSD_512_large', input_size=512, backbone_type='MLSD_large', num_samples=5, score_thr=0.5, dist_thr=10, topk=200)
Finetuned: TestConfig(model_path='/content/drive/MyDrive/Colab Notebooks/mlsd-for-duckitown/fine-tuned_model/M-LSD_512_large_ft_5_0001', input_size=512, backbone_type='MLSD_large', num_samples=5, score_thr=0.5, dist_thr=10, topk=200)


In [ ]:
import json
import os
from typing import List, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

import dataloader as dl
import load_model


def load_labels(label_path: str, image_dir: str) -> Tuple[List[str], List[List[List[float]]]]:
    if not os.path.isfile(label_path):
        raise FileNotFoundError(f"Label json not found: {label_path}")

    with open(label_path, "r", encoding="utf-8") as f:
        labels = json.load(f)

    image_paths = []
    lines_list = []
    for item in labels:
        filename = item.get("filename")
        lines = item.get("lines", [])
        if not filename:
            continue
        image_paths.append(os.path.join(image_dir, filename))
        lines_list.append(lines)

    if not image_paths:
        raise ValueError("No labels found in label json.")

    return image_paths, lines_list


def _nms_heatmap(score_map: np.ndarray, kernel: int = 3) -> np.ndarray:
    """3x3 non-maximum suppression (same idea as official M-LSD inference)."""
    dilated = cv2.dilate(score_map, np.ones((kernel, kernel), np.float32))
    return score_map * (score_map >= dilated - 1e-6)


def decode_lines_from_map(
    pred_map: np.ndarray,
    input_size: int,
    score_thr: float = 0.05,
    dist_thr: float = 20.0,
    topk: int = 200,
) -> Tuple[np.ndarray, np.ndarray]:
    """Decode TP lines from train_map (ch0=center, ch3:7=disp) via top-k + NMS."""
    vmap = pred_map[:, :, 3:7]
    h, w = pred_map.shape[:2]

    score_map = 1.0 / (1.0 + np.exp(-pred_map[:, :, 0]))
    score_map = _nms_heatmap(score_map)

    start = vmap[:, :, :2]
    end = vmap[:, :, 2:]
    dist_map = np.sqrt(np.sum((start - end) ** 2, axis=-1))

    flat_scores = score_map.reshape(-1)
    k = min(topk, flat_scores.size)
    topk_idx = np.argpartition(flat_scores, -k)[-k:]
    topk_idx = topk_idx[np.argsort(flat_scores[topk_idx])[::-1]]

    segments_list = []
    line_scores = []
    for idx in topk_idx:
        score = float(flat_scores[idx])
        if score <= score_thr:
            continue
        y, x = divmod(int(idx), w)
        distance = float(dist_map[y, x])
        if distance <= dist_thr:
            continue
        disp_x_start, disp_y_start, disp_x_end, disp_y_end = vmap[y, x]
        segments_list.append([
            x + disp_x_start,
            y + disp_y_start,
            x + disp_x_end,
            y + disp_y_end,
        ])
        line_scores.append(score)

    if not segments_list:
        return np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.float32)

    lines = np.array(segments_list, dtype=np.float32)
    line_scores = np.array(line_scores, dtype=np.float32)
    scale = float(input_size) / float(h)
    lines *= scale
    return lines, line_scores


def draw_lines_on_image(image_u8: np.ndarray, lines: np.ndarray, color=(0, 0, 255)) -> np.ndarray:
    """
    Draw detected lines on the image. Color is red by default. Lines are thick.
    """
    output = image_u8.copy()
    # Convert to 3-channel if needed
    if output.ndim == 2:
        output = cv2.cvtColor(output, cv2.COLOR_GRAY2BGR)
    elif output.shape[2] == 1:
        output = cv2.cvtColor(output, cv2.COLOR_GRAY2BGR)
    if lines.shape[0] == 0:
        return output
    for x1, y1, x2, y2 in lines:
        # Clip coordinates to image bounds
        h, w = output.shape[:2]
        x1 = np.clip(int(round(x1)), 0, w-1)
        y1 = np.clip(int(round(y1)), 0, h-1)
        x2 = np.clip(int(round(x2)), 0, w-1)
        y2 = np.clip(int(round(y2)), 0, h-1)
        cv2.line(
            output,
            (x1, y1),
            (x2, y2),
            color,
            2,  # line thickness
            cv2.LINE_AA,
        )
    return output


_OFFICIAL_MODEL_NAMES = {
    "M-LSD_320_tiny",
    "M-LSD_320_large",
    "M-LSD_512_tiny",
    "M-LSD_512_large",
}


def model_cfg_from_test_config(config: TestConfig) -> load_model.ModelConfig:
    model_name = os.path.basename(os.path.normpath(config.model_path))
    if model_name in _OFFICIAL_MODEL_NAMES:
        model_cfg = load_model.infer_config_from_path(config.model_path)
        model_cfg.batch_size = 1
        model_cfg.topk = config.topk
        return model_cfg

    return load_model.ModelConfig(
        input_size=config.input_size,
        map_size=config.input_size // 2,
        backbone_type=config.backbone_type,
        batch_size=1,
        topk=config.topk,
    )


def load_model_from_test_config(config: TestConfig) -> Tuple[tf.keras.Model, load_model.ModelConfig]:
    model_cfg = model_cfg_from_test_config(config)
    model = load_model.build_model(model_cfg, return_train_map=True)

    ckpt = tf.train.Checkpoint(step=tf.Variable(0, name="step"), model=model)
    manager = tf.train.CheckpointManager(ckpt, config.model_path, max_to_keep=1)
    if not manager.latest_checkpoint:
        raise FileNotFoundError(f"No checkpoint found in {config.model_path}")
    ckpt.restore(manager.latest_checkpoint).expect_partial()
    print(f"[*] load ckpt from {manager.latest_checkpoint} at step {ckpt.step.numpy()}.")
    return model, model_cfg


def run_inference_and_visualize(infer_config: TestConfig, title: str = "Inference") -> None:
    """Run test-set inference, print loss, and visualize detected lines."""
    print(f"\n{'=' * 60}")
    print(title)
    print(f"model_path: {infer_config.model_path}")
    print(f"{'=' * 60}")

    tf.keras.backend.clear_session()
    model, model_cfg = load_model_from_test_config(infer_config)

    test_image_paths, test_lines_list = load_labels(str(TEST_LABEL_PATH), str(TEST_IMAGE_DIR))
    test_dataset = dl.build_dataloader(
        image_paths=test_image_paths,
        lines_list=test_lines_list,
        batch_size=1,
        target_size=model_cfg.input_size,
        augment=False,
    )

    images_out = []
    loss_total = []
    loss_cls = []
    loss_reg = []
    line_counts = []

    for images, targets in test_dataset:
        preds = model(images, training=False)
        total_loss, cls_loss, reg_loss = compute_loss(targets, preds, infer_config)
        loss_total.append(float(total_loss.numpy()))
        loss_cls.append(float(cls_loss.numpy()))
        loss_reg.append(float(reg_loss.numpy()))

        image = images[0].numpy()
        if len(images_out) == 0:
            print(f"  debug: image range=[{image.min():.1f}, {image.max():.1f}] (正常: ~[0,255]、[0,1]なら dataloader.py が未修正)")
        image_u8 = image.clip(0, 255).astype(np.uint8)  # images are [0, 255]; backbone applies preprocess_input internally
        pred_map = preds[0].numpy()
        score_map = 1.0 / (1.0 + np.exp(-pred_map[:, :, 0]))
        lines, line_scores = decode_lines_from_map(
            pred_map,
            model_cfg.input_size,
            score_thr=infer_config.score_thr,
            dist_thr=infer_config.dist_thr,
            topk=infer_config.topk,
        )
        line_counts.append(len(lines))
        if len(images_out) == 0:
            print(
                f"  debug: max_score={score_map.max():.4f}, "
                f"pixels>{infer_config.score_thr}={(score_map > infer_config.score_thr).sum()}"
            )
        overlay = draw_lines_on_image(
            cv2.cvtColor(image_u8, cv2.COLOR_RGB2BGR),
            lines,
        )
        images_out.append(overlay)

        if len(images_out) >= infer_config.num_samples:
            break

    n = len(loss_total)
    print(f"Test loss ({n} samples):", end="")
    if loss_total:
        avg_total = sum(loss_total) / n
        avg_cls = sum(loss_cls) / n
        avg_reg = sum(loss_reg) / n
        print(f" total={avg_total:.4f} | cls={avg_cls:.4f} | reg={avg_reg:.4f}")
        print(f"Lines per image: {line_counts}")
    else:
        print(" No test samples found.")

    if images_out:
        fig, axes = plt.subplots(1, len(images_out), figsize=(4 * len(images_out), 4))
        if len(images_out) == 1:
            axes = [axes]
        for ax, img in zip(axes, images_out):
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.axis("off")
        fig.suptitle(title)
        plt.tight_layout()
        plt.show()

    del model
    tf.keras.backend.clear_session()


In [ ]:
run_inference_and_visualize(
    pretrained_test_config,
    title="Pretrained model (before fine-tuning)",
)

run_inference_and_visualize(
    finetuned_test_config,
    title="Fine-tuned model",
)